# MILU: Multilingual Indic Question Answering — Reference Solution

**Strategy**: domain-aware few-shot prompting with a multilingual instruction-tuned model.

- Reads from `./dataset/public/train.csv` and `./dataset/public/test.csv`
- Writes to `./working/submission.csv`
- Runtime: ~20 min on Kaggle T4 GPU with a local 3B model; ~10 min via API

## 0. Setup

In [ ]:
# Uncomment to install missing packages
# !pip install -q transformers accelerate sentencepiece protobuf

import os, re, random, json
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

# Fix all random seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

TRAIN_PATH  = Path('./dataset/public/train.csv')
TEST_PATH   = Path('./dataset/public/test.csv')
OUTPUT_PATH = Path('./working/submission.csv')
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print('Paths OK')
print(f'  train : {TRAIN_PATH}')
print(f'  test  : {TEST_PATH}')
print(f'  output: {OUTPUT_PATH}')

## 1. Load Data

In [ ]:
train_df = pd.read_csv(TRAIN_PATH, encoding='utf-8-sig')
test_df  = pd.read_csv(TEST_PATH,  encoding='utf-8-sig')

OPTION_COLS = ['option_a', 'option_b', 'option_c', 'option_d']

print(f'train: {len(train_df):,} rows  |  test: {len(test_df):,} rows')
print(f'columns: {list(train_df.columns)}')
print()
print('Language distribution (train):')
print(train_df['language'].value_counts().to_string())

In [ ]:
# Sanity check — all 11 languages present, no nulls in key columns
assert train_df['language'].nunique() == 11, 'Missing languages in train'
assert train_df[OPTION_COLS + ['question']].isna().sum().sum() == 0, 'Null in option/question columns'
assert train_df['answer'].isin(['A','B','C','D']).all(), 'Invalid answer values'

# Verify non-Latin scripts render correctly
print('Sample questions per script:')
for lang in ['hin', 'tam', 'ben', 'kan', 'eng']:
    row = train_df[train_df['language'] == lang].iloc[0]
    print(f'  [{lang}] {row["question"][:80]}')

## 2. Build Few-Shot Example Store

In [ ]:
# Index training examples by (language, domain) for fast retrieval
example_store = defaultdict(list)
lang_store    = defaultdict(list)

for _, row in train_df.iterrows():
    key = (row['language'], row.get('domain', ''))
    example_store[key].append(row)
    lang_store[row['language']].append(row)

def get_examples(language: str, domain: str, n: int = 3) -> list:
    """Return n few-shot examples for a language/domain pair."""
    pool = example_store.get((language, domain), lang_store.get(language, []))
    rng  = random.Random(hash(language + domain) % (2**32))
    return rng.sample(pool, min(n, len(pool)))

print(f'Store built: {len(example_store)} (language, domain) keys')
print(f'  e.g. Hindi/STEM: {len(example_store[("hin", "STEM")])} examples')

## 3. Prompt Construction

In [ ]:
def format_question(row, include_answer: bool = False) -> str:
    """Render a single MCQ row as prompt text."""
    lines = [f'Question: {row["question"]}']
    for letter, col in zip('ABCD', OPTION_COLS):
        lines.append(f'  {letter}. {row[col]}')
    if include_answer:
        lines.append(f'Answer: {row["answer"]}')
    else:
        lines.append('Answer:')
    return '\n'.join(lines)


def build_prompt(row, few_shot_rows: list) -> str:
    """Assemble a few-shot prompt for one test question."""
    lang_name = row.get('language_name', row['language'])
    domain    = row.get('domain', 'General')

    header = (
        f'You are an expert at answering multiple-choice questions in {lang_name}.\n'
        f'Domain: {domain}\n'
        'Respond with ONLY the letter of the correct answer: A, B, C, or D.\n\n'
    )
    examples = '\n\n'.join(format_question(r, include_answer=True) for r in few_shot_rows)
    test_q   = format_question(row, include_answer=False)

    return header + examples + ('\n\n' if examples else '') + test_q


# Preview
sample = train_df[train_df['language'] == 'hin'].iloc[3].to_dict()
shots  = get_examples('hin', sample.get('domain',''), n=2)
print(build_prompt(sample, shots))

## 4. Model Configuration

Set `BACKEND` to `'local'`, `'openai'`, or `'anthropic'`.

In [ ]:
BACKEND        = 'local'                       # 'local' | 'openai' | 'anthropic'
LOCAL_MODEL_ID = 'google/gemma-2-2b-it'        # ~5 GB, strong multilingual
# Alternatives: 'meta-llama/Llama-3.2-3B-Instruct', 'microsoft/Phi-3-mini-128k-instruct'

API_MODEL          = 'gpt-4o-mini'             # or 'claude-haiku-4-5-20251001'
OPENAI_API_KEY     = os.environ.get('OPENAI_API_KEY', '')
ANTHROPIC_API_KEY  = os.environ.get('ANTHROPIC_API_KEY', '')

FEW_SHOT_N     = 3    # examples per question
MAX_NEW_TOKENS = 5    # single letter output
BATCH_SIZE     = 8    # local model batch size

print(f'Backend: {BACKEND} / Model: {LOCAL_MODEL_ID if BACKEND == "local" else API_MODEL}')

In [ ]:
model, tokenizer = None, None

if BACKEND == 'local':
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    torch.manual_seed(SEED)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Device: {device}')

    tokenizer = AutoTokenizer.from_pretrained(
        LOCAL_MODEL_ID, trust_remote_code=True, padding_side='left'
    )
    tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        LOCAL_MODEL_ID,
        torch_dtype=torch.bfloat16 if device == 'cuda' else torch.float32,
        device_map='auto' if device == 'cuda' else None,
        trust_remote_code=True,
    ).eval()
    print('Model loaded.')

## 5. Inference

In [ ]:
ANSWER_RE = re.compile(
    r'\b([ABCD])\b|\(([ABCD])\)|([ABCD])[.):]|answer\s*(?:is)?\s*:?\s*([ABCD])\b',
    re.IGNORECASE
)

def extract_answer(text: str) -> str:
    text = text.strip()
    if text.upper() in ('A', 'B', 'C', 'D'):
        return text.upper()
    m = ANSWER_RE.search(text)
    if m:
        return next(g for g in m.groups() if g).upper()
    for ch in text.upper():
        if ch in ('A', 'B', 'C', 'D'):
            return ch
    return 'A'  # fallback

# Quick tests
cases = ['B', '(C)', 'The answer is D.', 'Answer: A', 'B) is right', 'xyz']
for c in cases:
    print(f'  {c!r:35s} → {extract_answer(c)}')

In [ ]:
def predict_local_batch(prompts: list) -> list:
    import torch
    inputs = tokenizer(
        prompts, return_tensors='pt', padding=True,
        truncation=True, max_length=1024
    ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False, temperature=None, top_p=None,
            pad_token_id=tokenizer.pad_token_id,
        )
    input_len = inputs['input_ids'].shape[1]
    return [
        tokenizer.decode(out[input_len:], skip_special_tokens=True)
        for out in outputs
    ]

def predict_openai(prompt: str) -> str:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    r = client.chat.completions.create(
        model=API_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=5, temperature=0, seed=SEED,
    )
    return r.choices[0].message.content.strip()

def predict_anthropic(prompt: str) -> str:
    import anthropic
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    r = client.messages.create(
        model=API_MODEL, max_tokens=5,
        messages=[{'role': 'user', 'content': prompt}],
    )
    return r.content[0].text.strip()

print('Inference functions defined.')

## 6. Per-Language Validation on Train Split

In [ ]:
from sklearn.model_selection import train_test_split

pool_df, val_df = train_test_split(
    train_df, test_size=0.15,
    stratify=train_df['language'], random_state=SEED
)

# Rebuild few-shot store from pool only
pool_store = defaultdict(list)
pool_lang  = defaultdict(list)
for _, row in pool_df.iterrows():
    pool_store[(row['language'], row.get('domain',''))].append(row)
    pool_lang[row['language']].append(row)

def get_pool_examples(language, domain, n=3):
    pool = pool_store.get((language, domain), pool_lang.get(language, []))
    rng  = random.Random(hash(language + domain) % (2**32))
    return rng.sample(pool, min(n, len(pool)))

# Subsample validation for speed (30 per language)
val_sample = (
    val_df.groupby('language', group_keys=False)
    .apply(lambda g: g.sample(min(30, len(g)), random_state=SEED))
    .reset_index(drop=True)
)
print(f'Validation sample: {len(val_sample)} rows across {val_sample["language"].nunique()} languages')

In [ ]:
val_preds = []
rows = val_sample.to_dict('records')

if BACKEND == 'local' and model is not None:
    for i in range(0, len(rows), BATCH_SIZE):
        batch   = rows[i:i + BATCH_SIZE]
        prompts = [build_prompt(r, get_pool_examples(r['language'], r.get('domain',''), FEW_SHOT_N)) for r in batch]
        raws    = predict_local_batch(prompts)
        val_preds.extend(extract_answer(r) for r in raws)
elif BACKEND == 'openai':
    for r in rows:
        raw = predict_openai(build_prompt(r, get_pool_examples(r['language'], r.get('domain',''), FEW_SHOT_N)))
        val_preds.append(extract_answer(raw))
elif BACKEND == 'anthropic':
    for r in rows:
        raw = predict_anthropic(build_prompt(r, get_pool_examples(r['language'], r.get('domain',''), FEW_SHOT_N)))
        val_preds.append(extract_answer(raw))
else:
    print('No backend — using mock random predictions for demonstration.')
    val_preds = list(np.random.choice(['A','B','C','D'], size=len(rows)))

val_sample = val_sample.copy()
val_sample['predicted'] = val_preds
val_sample['correct']   = val_sample['answer'] == val_sample['predicted']

overall_acc = val_sample['correct'].mean()
print(f'Overall validation accuracy: {overall_acc:.4f} ({overall_acc*100:.1f}%)')

In [ ]:
lang_acc = (
    val_sample.groupby('language')['correct']
    .agg(accuracy='mean', n='count')
    .sort_values('accuracy', ascending=False)
    .assign(accuracy=lambda d: d['accuracy'].round(4))
)
print('Per-language accuracy:')
print(lang_acc.to_string())
print()

# Identify weakest language — can be used to allocate more few-shot examples
weakest = lang_acc['accuracy'].idxmin()
print(f'Weakest language: {weakest} ({lang_acc.loc[weakest, "accuracy"]:.4f})')
print('Consider increasing FEW_SHOT_N or using domain-specific prompts for this language.')

## 7. Generate Test Predictions

In [ ]:
test_rows  = test_df.to_dict('records')
final_preds = []  # list of {question_id, answer}

if BACKEND == 'local' and model is not None:
    for i in range(0, len(test_rows), BATCH_SIZE):
        batch   = test_rows[i:i + BATCH_SIZE]
        prompts = [build_prompt(r, get_examples(r['language'], r.get('domain',''), FEW_SHOT_N)) for r in batch]
        raws    = predict_local_batch(prompts)
        for r, raw in zip(batch, raws):
            final_preds.append({'question_id': r['question_id'], 'answer': extract_answer(raw)})
        if (i // BATCH_SIZE) % 20 == 0:
            pct = 100 * (i + len(batch)) / len(test_rows)
            print(f'  {i + len(batch):,}/{len(test_rows):,}  ({pct:.1f}%)')

elif BACKEND == 'openai':
    for i, r in enumerate(test_rows):
        raw = predict_openai(build_prompt(r, get_examples(r['language'], r.get('domain',''), FEW_SHOT_N)))
        final_preds.append({'question_id': r['question_id'], 'answer': extract_answer(raw)})
        if i % 100 == 0: print(f'  {i}/{len(test_rows)}…')

elif BACKEND == 'anthropic':
    for i, r in enumerate(test_rows):
        raw = predict_anthropic(build_prompt(r, get_examples(r['language'], r.get('domain',''), FEW_SHOT_N)))
        final_preds.append({'question_id': r['question_id'], 'answer': extract_answer(raw)})
        if i % 100 == 0: print(f'  {i}/{len(test_rows)}…')

else:
    print('Demo mode: random predictions.')
    rng = random.Random(SEED)
    for r in test_rows:
        final_preds.append({'question_id': r['question_id'], 'answer': rng.choice(['A','B','C','D'])})

print(f'Generated {len(final_preds):,} predictions.')

## 8. Write Submission

In [ ]:
submission = pd.DataFrame(final_preds, columns=['question_id', 'answer'])

# Validate
assert list(submission.columns) == ['question_id', 'answer']
assert submission['answer'].isin(['A','B','C','D']).all(), 'Invalid answer values'
assert submission['question_id'].nunique() == len(submission), 'Duplicate question_ids'
assert len(submission) == len(test_df), f'Row count mismatch: {len(submission)} vs {len(test_df)}'

submission.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')

print(f'Submission saved: {OUTPUT_PATH}')
print(f'Rows: {len(submission):,}')
print('Answer distribution:')
print(submission['answer'].value_counts().to_string())

## Summary

| Step | What we did |
|------|-------------|
| Data loading | UTF-8 CSVs, 11 languages, flat option columns |
| Few-shot store | Indexed by `(language, domain)` for domain-aware retrieval |
| Prompting | Language name + domain in header; 3 in-language examples |
| Answer extraction | Regex with fallback — handles 7+ output formats |
| Validation | Per-language accuracy on 15% holdout; weakest language flagged |
| Submission | `question_id`, `answer` — validated before write |

**Expected score**: ~0.60–0.70 with a 3B multilingual model; ~0.74+ with GPT-4o.

**To improve further**:
- LoRA finetune on `train.csv` (+5–10% on low-resource languages)
- Ensemble with majority vote over 5 temperature>0 passes (+2–4%)
- Chain-of-thought for STEM and Law & Governance domains (+3–6%)